In [ ]:
import pandas as pd
import folium
import json
from folium import Choropleth

main_df = pd.read_csv('main_df')
msc_borders_clean = pd.read_json("msc_borders_clean")

# Выбираем нужные данные
taxi_aggregated = main_df.groupby('district_name')['n_taxi_start'].sum().reset_index()
# Преобразуем в словарь для быстрого поиска по названию района
taxi_aggregated = dict(zip(taxi_aggregated['district_name'], taxi_aggregated['n_taxi_start']))

# Добавляем количесво поездок в свойства районов GeoJSON
for feature in msc_borders_clean['features']:
    district_name = feature['properties']['name']
    feature['properties']['n_taxi_start'] = taxi_aggregated.get(district_name)

# Создаем карту Москвы
moscow_lat, moscow_lng = 55.751244, 37.618423
m = folium.Map(location=[moscow_lat, moscow_lng], zoom_start=10, tiles='Cartodb Positron')

# Добавляем хороплет-карту с количеством поездок
choropleth = Choropleth(
    geo_data=msc_borders_clean,
    data=taxi_aggregated,
    columns=['district_name', 'n_taxi_start'],
    key_on='feature.properties.name',  # Связываем с name в GeoJSON
    fill_color='YlOrRd',  # Цветовая гамма (желтый -> оранжевый -> красный)
    fill_opacity=0.8,
    line_opacity=0.2,
    legend_name='Количество поездок за год'
).add_to(m)

# Добавляем всплывающую информацию при наведении на район
folium.GeoJson(
    msc_borders_clean,
    style_function=lambda feature: {
        "fillColor": "transparent",  # Не перезаписываем цвет
        "color": "black",
        "weight": 1,
        "fillOpacity": 0
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["name", "n_taxi_start"],
        aliases=["Район", "Количество поездок"],
        localize=True,
        sticky=False
    )
).add_to(m)

from branca.element import Element
# CSS для скрытия надписи Leaflet
css = """
<style>
.leaflet-control-attribution {
    display: none !important;
}
</style>
"""
# Добавляем стиль на карту
m.get_root().html.add_child(Element(css))
# Отображаем карту

m